# Manipulations des règles

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle,re,features,pyperclip, operator
import networkx as nx
debug=False

In [ ]:
%store -r ordStemCells
ordStemCells

## Préparatifs

### Transformation du fichier des traits des phonèmes en tableau LaTeX

In [140]:
nFeatures='/Users/gilles/Github/SWIM/ParadigmGeneration/L4L/bdlexique.ini'
dfFeatures = pd.read_csv(nFeatures, sep="|", skiprows=4,index_col=0)
dfFeatures

,+son,-son,+syl,-syl,+cons,-cons,+ant,-ant,+cor,-cor,...,-nas,+lat,-lat,+cont,-cont,+voice,-voice,+strid,-strid,Unnamed: 31
,,,,,,,,,,,,,,,,,,,,,
p,,X,,X,X,,X,,,X,...,X,,X,,X,,X,,X,NaN
t,,X,,X,X,,X,,X,,...,X,,X,,X,,X,,X,NaN
k,,X,,X,X,,,X,,X,...,X,,X,,X,,X,,X,NaN
b,,X,,X,X,,X,,,X,...,X,,X,,X,X,,,X,NaN
d,,X,,X,X,,X,,X,,...,X,,X,,X,X,,,X,NaN
g,,X,,X,X,,,X,,X,...,X,,X,,X,X,,,X,NaN
f,,X,,X,X,,X,,,X,...,X,,X,X,,,X,X,,NaN
s,,X,,X,X,,X,,X,,...,X,,X,X,,,X,X,,NaN
S,,X,,X,X,,,X,X,,...,X,,X,X,,,X,X,,NaN


In [142]:
df_initial=dfFeatures
print("--- DataFrame initial ---")
print(df_initial)

# 2. Liste de vos 16 traits (sans le prefixe '+' ou '-')
# Remplacez cette liste par les vrais noms de vos 16 traits
liste_traits = ["son", "syl", "cons", "ant","cor", "back", "high", "low", "round","ATR", "nas", "lat", "cont", "voice", "strid",]

# 3. Transformation
df_final = pd.DataFrame(index=df_initial.index)

for trait in liste_traits:
    col_plus = f"+{trait}"
    col_moins = f"-{trait}"

    # Conditions : vérifie où se trouvent les 1 (ou True)
    conditions = [
        df_initial[col_plus].str.strip() == "X",
        df_initial[col_moins].str.strip() == "X",
    ]
    choix = ["+", "-"]

    # np.select attribue '+' si col_plus==1, '-' si col_moins==1, sinon None
    df_final[trait] = np.select(conditions, choix, default=None)

print("\n--- DataFrame final ---")
print(df_final.to_latex())

--- DataFrame initial ---
      +son  -son  +syl  -syl  +cons  -cons  +ant  -ant  +cor  -cor  ...  -nas  \
                                                                    ...         
p            X           X      X            X                 X    ...   X     
t            X           X      X            X           X          ...   X     
k            X           X      X                  X           X    ...   X     
b            X           X      X            X                 X    ...   X     
d            X           X      X            X           X          ...   X     
g            X           X      X                  X           X    ...   X     
f            X           X      X            X                 X    ...   X     
s            X           X      X            X           X          ...   X     
S            X           X      X                  X     X          ...   X     
v            X           X      X            X                 X    ...   X     
z 

### Préparation du codage des phonèmes

In [2]:
features.add_config(nFeatures)
fs=features.FeatureSystem('phonemes')
# fs.context.objects


In [3]:
# traduire SAMPA-BDLex en API

def sampa2api(sampa):
    api=sampa
    api=api.replace(u'n"',u'n') 
    api=api.replace(u't"',u't') 
    api=api.replace(u'z"',u'z') 
    api=api.replace(u'R"',u'ʁ') 
    api=api.replace(u'p"',u'p') 
    api=api.replace(u'S',u'ʃ') 
    api=api.replace(u'Z',u'ʒ')
    api=api.replace(u'N',u'ŋ')
    api=api.replace(u'J',u'ɲ')
    # api=api.replace(u'r',u'ʁ') 
    api=api.replace(u'H',u'ɥ')
    api=api.replace(u'E',u'ɛ')
    api=api.replace(u'2',u'ø')
    api=api.replace(u'9',u'œ')
    api=api.replace(u'6',u'ə')
    api=api.replace(u'O',u'ɔ')
    api=api.replace(u'è',u'e')   
    api=api.replace(u'ò',u'o')    
    api=api.replace(u'â',u'ɑ̃')   
    api=api.replace(u'ê',u'ɛ̃')   
    api=api.replace(u'û',u'œ̃')  
    api=api.replace(u'ô',u'ɔ̃')       
    api=api.replace(u'@',u'ə')
    api=api.replace(u'R',u'ʁ') 
    return api

### Préparation des structures pour les règles

In [4]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n, file=logfile)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

    
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=True):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug:
                    print (forme, file=logfile)
                    print ("pas de classe",idClasseForme, file=logfile)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)), file=logfile)
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme, file=logfile) 
                print ("pas de patron", file=logfile)
        return sortieForme
        

## Lecture du fichier de règles

In [5]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
rulesFile="vlexique2-Total-Regles.pkl"
fRules=repFiles+rulesFile
with open(fRules, 'rb') as input:
    regles = pickle.load(input)
# resultatsLecture[('fi1P', 'ai1P')].classeCF 

pFreq={}
for k in regles:
    for p in regles[k].patrons:
        if p not in pFreq:
            pFreq[p]=0
        pFreq[p]+=1

In [263]:
regles[("inf","pP")].classeCF

{'E-â, E-yâ': {'E-â': 4464, 'E-yâ': 1},
 'r-sâ, r-â, ir-â, r-jâ, r-zâ, r-vâ, ir-jâ': {'r-sâ': 286,
  'ir-â': 43,
  'r-zâ': 32,
  'r-vâ': 11,
  'r-jâ': 2,
  'ir-jâ': 2},
 'r-sâ, r-â, r-jâ, tr-sâ, r-zâ, r-vâ': {'r-â': 27, 'tr-sâ': 19},
 'r-sâ, r-â, udr-Olvâ, r-jâ, r-zâ, r-vâ, dr-zâ, dr-lâ': {'udr-Olvâ': 3,
  'dr-zâ': 3,
  'dr-lâ': 3},
 'r-sâ, r-â, ir-â, r-jâ, r-zâ, r-vâ, 9..r-O..sâ, ir-jâ': {'ir-â': 31,
  'r-sâ': 9,
  'r-zâ': 3,
  '9..r-O..sâ': 2},
 'r-sâ, r-â, r-jâ, r-zâ, r-vâ, Er-9zâ': {'r-jâ': 7, 'r-zâ': 4, 'Er-9zâ': 6},
 'r-sâ, r-â, ir-â, r-jâ, r-zâ, r-vâ, E.ir-i.â, ir-jâ': {'r-sâ': 51,
  'ir-â': 8,
  'r-zâ': 5,
  'E.ir-i.â': 1},
 'E-â, HE-â, E-yâ': {'E-â': 58, 'HE-â': 8},
 'r-sâ, r-â, r-jâ, êdr-aJâ, êdr-EJâ, r-zâ, r-vâ, dr-zâ, dr-lâ': {'êdr-aJâ': 8,
  'êdr-EJâ': 20},
 'E-â, E-yâ, wE-â': {'E-â': 24, 'wE-â': 10},
 'r-sâ, r-â, r-jâ, war-â, war-Ejâ, war-yvâ, r-zâ, r-vâ, vwar-Sâ, war-Eâ': {'war-â': 13,
  'r-jâ': 6},
 'r-sâ, r-â, r-jâ, âdr-9nâ, r-zâ, r-vâ, dr-zâ, dr-lâ': {'r-â': 28,
  'âd

In [273]:
nb=0
entropies={}
for c,cf in regles[("inf","pP")].classeCF.items():
    v=list(cf.values())
    s=sum(v)
    nb+=s
    entropies[c]=(s,entropy(list(cf.values())))
entropie=0
for (s,e) in entropies.values():
    entropie+=s/nb*e
nb, entropie

(5267, 0.11337514501202287)

### Liste des transformations par fréquence

In [ ]:
dict(sorted(pFreq.items(), key=lambda item: item[1],reverse=True))

### Affichage des règles au format LaTeX

In [245]:
def stripC(text):
    result=text
    joker=r"\^\(\.\*"
    m=re.match(joker+r"(\w+)?(\[[^)]*)\)",text)
    mm=re.match(joker+r"(\w+)",text)
    mmm=re.match(r"\^(\w+)",text)
    if m:
        result="".join([g for g in m.groups() if g is not None])
    elif mm:
        result=mm.group(1)
    elif mmm:
        result=mmm.group(1)
    return sampa2api(result)
    
for k,v in regles[("inf","pP")].patrons.items():
    print(k,v)
    a,b = k.split("-")
    c=v.rsplit(a,1)
    print("\\ex \\phonc{\\textipan{%s}}{\\textipan{%s}}{\\textipan{%s} \\phold{} \\#}"%(sampa2api(a),sampa2api(b),stripC(c[0])))
    # print()

E-â ^(.*[ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwHE96Oêûô])E$
\ex \phonc{\textipan{ɛ}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəɔɛ̃œ̃ɔ̃]} \phold{} \#}
r-sâ ^(.*[ptkbdgfsSvzZmnJNjlrwHE96aOêûâô])ir$
\ex \phonc{\textipan{r}}{\textipan{sɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjlrwɥɛœəaɔɛ̃œ̃ɑ̃ɔ̃]} \phold{} \#}
r-â ^(.*[mnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZjrwHiyEe926uOo])r$
\ex \phonc{\textipan{r}}{\textipan{ɑ̃}}{\textipan{[mnɲŋjlrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒjrwɥiyɛeœøəuɔo]} \phold{} \#}
udr-Olvâ ^([bdvzrE6a][pbjiEe][sz])udr$
\ex \phonc{\textipan{udr}}{\textipan{ɔlvɑ̃}}{\textipan{^([bdvzrɛəa][pbjiɛe][sz])} \phold{} \#}
ir-â ^(.*[ptkbdgfsSvzZmnJNjrwHiyEe926auOoêûâô][fvjrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjr])ir$
\ex \phonc{\textipan{ir}}{\textipan{ɑ̃}}{\textipan{[ptkbdgfsʃvzʒmnɲŋjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][fvjrwɥiyɛeœøəauɔoɛ̃œ̃ɑ̃ɔ̃][ptkbdgfsʃvzʒmnɲŋjr]} \phold{} \#}
r-jâ ^(.*[ptbdfsvzr][jrwH][iEea])r$
\ex \phonc{\textipan{r}}{

## Génération des formes candidates

In [ ]:
paires=regles.keys()
cases=set(k for k,v in paires)
case1="pi3S"
case2="fi3S"
forme1="brwa"
forme2="brwara"
c1={}
c2={}
common={}
vals={}
for case in cases:
    cVals=vals[case]={}
    c1[case]=regles[(case1,case)].sortirForme(forme1,contextFree=False)
    c2[case]=regles[(case2,case)].sortirForme(forme2,contextFree=False)
    print(case1,case,c1[case])
    print(case2,case,c2[case])

    common[case]=c1[case].keys() & c2[case].keys()
    
    for k in common[case]:
        cVals[k]=c1[case][k]+c2[case][k]
    tVals=sum(cVals.values())
    print()
    for k,v in cVals.items():
        cVals[k]=cVals[k]/tVals
        print(k, f"{cVals[k]:.2f}")    
    print()
    print("========================")



In [ ]:
vals

In [ ]:
print (case1, forme1, case2, forme2)
print()
for k1,v1 in vals.items():
    print(k1,end=" : ")
    for k2,v2 in dict(sorted(v1.items(), key=lambda x: x[1], reverse=True)).items():
        print(f"{v2:.0%}",k2,end=", ") 
    print()

In [ ]:
vals.get("pi3S", {})
# vals[temps_finis[t]+personnes_finies[p]]

# Génération tableau de candidates

In [ ]:
# Structure des clés / cases
personnes_finies = {"1SG":"1S", "2SG":"2S", "3SG":"3S", "1PL":"1P", "2PL":"2P", "3PL":"3P"}
temps_finis = {"présent":"pi", "imparfait":"ii", "passé":"ai", "futur":"fi", "subj. prés.":"ps", "subj. imparf.":"is", "conditionnel":"pc", "impératif":"pI"}

maxT={}
for t in temps_finis.values():
    maxT[t]=max([len(v) for k,v in vals.items() if t in k]) 
for t in ["inf","pP","ppMS","ppMP","ppFS","ppFP"]:
    maxT[t]=max([len(v) for k,v in vals.items() if t in k]) 

print(maxT)

In [ ]:
# On suppose que `vals` est structuré ainsi : vals[(temps, personne)] = {forme: pourcentage}

def textipa(text):
    return r"\textipan{%s}"%sampa2api(text)

def multirow(text,n):
    if n == 1:
        return text
    else:
        return r"\multirow{%d}{*}{%s}"%(n,text)

def escape_latex(text):
    """Échappe le signe % pour qu'il soit valide en LaTeX."""
    return str(text).replace('%', r'\%')

def format_cellule(dict_formes):
    """Formate le contenu d'une cellule avec retours à la ligne si multiple."""
    if not dict_formes:
        return "--"
    
    # Tri décroissant selon le pourcentage/poids
    items = sorted(dict_formes.items(), key=operator.itemgetter(0, 1), reverse=False)
    
    # Cas à 100% avec une seule forme (affiché sans pourcentage dans l'image si nécessaire, ou avec)
    # Dans l'image : si 100%, il est écrit "100% abwa" (ou juste en gras selon le cas).
    
    lines = []
    for forme, pct in items:
        if isinstance(pct, (int, float)):
            pct_str = f"{pct:.0%}".replace('%', r'\%')
        else:
            pct_str = str(pct)
        lines.append(f"{pct_str} {textipa(forme)}")
    
    # Si la cellule contient plusieurs formes, on utilise \shortstack pour les empiler
    if len(lines) > 1:
        return r"\newline ".join(lines)
    elif len(lines) == 1:
        return multirow(lines[0],maxT[tt])
    return "--"

In [ ]:
# --- GÉNÉRATION DU CODE LATEX ---

latex_code = []

latex_code.append(r"\begin{tabular}[t]{@{\hspace{.5ex}}l" + r"@{\hspace{.5ex}}p{13ex}" * len(personnes_finies) + r"@{\hspace{.5ex}}}")
latex_code.append(r"\toprule")
latex_code.append(r"\textbf{formes finies} & " + " & ".join([f"\\textsc{{{p.lower()}}}" for p in personnes_finies]) + r" \\")
latex_code.append(r"\midrule")

for t,tt in temps_finis.items():
    row=[multirow(t,maxT[tt])]
    for p,pp in personnes_finies.items():
        cell_data = vals.get(tt+pp, {})
        row.append(format_cellule(cell_data))
    
    
    
    # Lignes de séparation de sections comme dans l'image
    if t in ["futur", "subj. imparf.", "conditionnel"]:
        line_str = " & ".join(row)+r"\\"+"\n"+r"\midrule"
    else:
        if maxT[tt]>1:
            saut="[12pt]"
        else:
            saut="[3pt]"
        line_str = " & ".join(row) + r" \\"+saut
        
    latex_code.append(line_str)

latex_code.append(r"\bottomrule")
latex_code.append(r"\end{tabular}")

# ESPACEMENT ENTRE LES DEUX TABLEAUX
latex_code.append(r"\hfill")

# TABLEAU 2 : Formes non-finies (à droite)
latex_code.append(r"\begin{tabular}[t]{@{\hspace{.5ex}}l@{\hspace{.5ex}}p{1.6cm}@{\hspace{.5ex}}}")
latex_code.append(r"\toprule")
latex_code.append(r"\multicolumn{2}{c}{\textbf{formes non-finies}} \\")
latex_code.append(r"\midrule")

# Exemples de formes non-finies
formes_non_finies = [
    (multirow("infinitif",maxT["inf"]), vals.get("inf", {})),
    (multirow("part. prés.",maxT["pP"]), vals.get("pP", {})),
    (multirow(r"\textsc{m.sg}",maxT["ppMS"]), vals.get("ppMS", {})),
    (multirow(r"\textsc{m.pl}",maxT["ppMP"]), vals.get("ppMP", {})),
    (multirow(r"\textsc{f.sg}",maxT["ppFS"]), vals.get("ppFS", {})),
    (multirow(r"\textsc{f.pl}",maxT["ppFP"]), vals.get("ppFP", {})),
]

for label, cell_data in formes_non_finies:
    cell_formatted = format_cellule(cell_data)
    latex_code.append(f"{label} & {cell_formatted} \\\\")
    if "infinitif" in label:
        latex_code.append(r"\midrule")
    elif "part. prés." in label:
        latex_code.append(r"\midrule")
        latex_code.append(r"\multicolumn{2}{c}{part. passé} \\")

latex_code.append(r"\bottomrule")
latex_code.append(r"\end{tabular}")

# Affichage du résultat
print("\n".join(latex_code))
pyperclip.copy("\n".join(latex_code))

# Faire le graphe des cliques

In [ ]:
transformations=nx.DiGraph()
paires={}
for c in ordStemCells:
    # print(c,vals[c])
    for cc in ordStemCells:
        for k in vals[c]:
            kDist=regles[(c,cc)].sortirForme(k,contextFree=False)
            commonDist=kDist.keys() & vals[cc].keys()
            for kk in commonDist:
                transformations.add_edge(c+"-"+k,cc+"-"+kk, weight=kDist[kk])

In [ ]:

def to_undirected_mean_weight(G: nx.DiGraph, weight_key="weight"):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))

    for u, v, data in G.edges(data=True):
        a, b = sorted((u, v))  # paire non orientée

        if H.has_edge(a, b):
            # on met à jour la moyenne en mode somme/nb
            H[a][b]["_sum"] += data.get(weight_key, 1)
            H[a][b]["_n"] += 1
        else:
            H.add_edge(a, b,
                       weight=data.get(weight_key, 1),
                       _sum=data.get(weight_key, 1),
                       _n=1)

    # finaliser la moyenne
    for a, b in list(H.edges()):
        H[a][b]["weight"] = H[a][b]["_sum"] / H[a][b]["_n"]
        del H[a][b]["_sum"]
        del H[a][b]["_n"]

    return H

relations = to_undirected_mean_weight(transformations)

cliques = list(nx.find_cliques(relations))
for clique in cliques:
    print(len(clique),", ".join(sorted(clique)))
    pClique=[]
    for node1 in clique:
        for node2 in clique:
            pRelation=relations[node1][node2]["weight"]
            pClique.append(pRelation)
            # print(node1,node2,pRelation)
    print(np.mean(pClique))

In [ ]:
for clique in cliques:
    print(len(clique),", ".join(sorted(clique)))
    # nodes1=[n for n in clique if "pi3S" in n or "fi3S" in n]
    pClique=[]
    for node1 in clique:
        for node2 in clique:
            pRelation=relations[node1][node2]["weight"]
            pClique.append(pRelation)
            # print(node1,node2,pRelation)
    print(np.mean(pClique))

In [ ]:
# (set(cliques[0]) & 
(set(cliques[0]) - set(cliques[1]))

In [ ]:
subgraph = relations.subgraph(cliques[0])
plt.figure(figsize=(50, 50))
pos = nx.circular_layout(subgraph)
nx.draw_networkx(
        subgraph,
        pos=pos,
        node_color="violet",
        node_size=50,
        font_size=10,
        edge_color="orange",
        width=2,
    )
# plt.savefig("TEMP.png",dpi=300,bbox_inches="tight")

In [ ]:
regles[("inf","ai2S")].sortirForme("abwar",contextFree=False)

In [ ]:
import itertools
import matplotlib.pyplot as plt
import networkx as nx

# Definition des trois listes de nœuds
liste1 = cliques[0]

liste2 = cliques[1]

liste3 = cliques[2]

listes = [liste1, liste2, liste3]

# Création du graphe non-orienté
G = nx.Graph()

# Ajout des arêtes : pour chaque liste, on connecte tous les nœuds deux à deux
for liste in listes:
    # Nettoyage des espaces insécables éventuels dans les chaînes
    noeuds_nettoyes = [noeud.strip() for noeud in liste]
    G.add_edges_from(itertools.combinations(noeuds_nettoyes, 2))

# Visualisation du graphe
plt.figure(figsize=(14, 10))

# Disposition des nœuds (spring layout donne un bon rendu pour les cliques)
pos = nx.spring_layout(G, k=0.5, seed=42)

# Dessin des nœuds, des arêtes et des étiquettes
nx.draw_networkx_nodes(G, pos, node_size=1500, node_color="skyblue", alpha=0.9)
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5, edge_color="gray")
nx.draw_networkx_labels(G, pos, font_size=8, font_family="sans-serif")

plt.title("Graphe non-orienté interconnecté", fontsize=14)
plt.axis("off")
plt.tight_layout()

# Affichage
plt.show()

# nx.nx_pydot.write_dot(G, "mon_graphe.dot")

In [ ]:
for k,v in regles[("inf","pP")].patrons.items():
    print(k,v)
    m=re.split("(\[[^\]]*\])",v)
    print(k,end=" : ")
    for e in m:
        if e.startswith("[") and e.endswith("]"):
            print(fs.lattice[e[1:-1]].intent,end="")
        elif e!="":
            print(e,end="")
    print()
    print()

In [ ]:
regles[("ai3S","ii3S")].sortirForme("bry",False)


# Extraction des règles

### préparatifs extraction

In [6]:
def diff(mot1,mot2):
    result=[]
    diff1=""
    diff2=""
    same=""
    vide="."
    lmax=max(len(mot1),len(mot2))
    lmin=min(len(mot1),len(mot2))
    for index in range(lmax):
        if index < lmin:
            if mot1[index]!=mot2[index]:
                diff1+=mot1[index]
                diff2+=mot2[index]
                same+=vide
            else:
                same+=mot1[index]
                diff1+=vide
                diff2+=vide
        elif index < len(mot1):
            diff1+=mot1[index]
        elif index < len(mot2):
            diff2+=mot2[index]
    diff1=diff1.lstrip(".")
    diff2=diff2.lstrip(".")
#    return (same,diff1,diff2,diff1+"_"+diff2)
    return (diff1+"-"+diff2)

In [7]:
def patron2regexp(morceaux):
    result="^"
    for morceau in morceaux:
        if morceau=="*":
            result+="(.*)"
        elif len(morceau)>1:
            result+="(["+morceau+"])"
        else:
            result+=morceau
    result+="$"
    result=result.replace(")(","")
    return result

In [8]:
class formesPatron:
    '''
    Accumulateur de formes correspondant à un patron pour calcul de la Généralisation Minimale (cf. MGL)
    '''
    def __init__(self):
        self.formes=[]

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterForme(self,forme):
        self.formes.append(forme)
        
    def calculerGM(self):
        minLongueur=len(min(self.formes, key=len))
        maxLongueur=len(max(self.formes, key=len))
        if debug: 
            print (minLongueur, maxLongueur, file=logfile)
            print (minLongueur, maxLongueur)
        positions=[]
        if maxLongueur>minLongueur:
            positions.append("*")
        for i in range(minLongueur, 0, -1):
            phonemes=set([x[-i] for x in self.formes])
            # print("phonemes",phonemes)
            if debug: 
                print (phonemes, file=logfile)
                print (phonemes)
            if "." in phonemes:
                positions.append(".")
            else:
                positions.append("".join(fs.lattice[phonemes].extent))
        return patron2regexp(positions)

class pairePatrons:
    '''
    Accumulateur de triplets (f1,f2,patron) correspondant à une paire pour calcul des Généralisations Minimales (cf. MGL)
    '''
    def __init__(self,case1,case2):
        self.patrons1={}
        self.patrons2={}
        self.case1=case1
        self.case2=case2

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterFormes(self,forme1,forme2,patron):
#        print (forme1,forme2,patron, file=logfile)
        patron12=patron
        (pat1,pat2)=patron.split("-")
        patron21=pat2+"-"+pat1
#        print (patron12,patron21, file=logfile)
        if not patron12 in self.patrons1:
            self.patrons1[patron12]=formesPatron()
        self.patrons1[patron12].ajouterForme(forme1)
        if not patron21 in self.patrons2:
            self.patrons2[patron21]=formesPatron()
        self.patrons2[patron21].ajouterForme(forme2)
        
        
    def calculerGM(self):
        resultat1={}
        for patron in self.patrons1:
            if debug: 
                print ("patron1", patron, file=logfile)
                print ("patron1", patron)
            resultat1[patron]=self.patrons1[patron].calculerGM()
        resultat2={}
        for patron in self.patrons2:
            if debug: 
                print ("patron2", patron, file=logfile)
                print ("patron2", patron)
            resultat2[patron]=self.patrons2[patron].calculerGM()
        return (resultat1,resultat2) 

### Extractions des règles

In [51]:
from itertools import combinations

In [187]:
def feat2phonrule(text):
    feat=fs.lattice[text[1:-1]].intent
    return r"\phonfeat[l]{"+r"\\".join(feat)+"}"

def translateRule(motif,context):
    result=[]
    m=[chunk for chunk in re.split("(\[[^\]]*\])",context) if chunk!=""]
    for e in m:
        if e.startswith("[") and e.endswith("]"):
            # result.append(fs.lattice[e[1:-1]].intent)
            result.append(feat2phonrule(e))
        elif e!="" and e!=None:
            result.append(e)
    # print(motif,context," ".join(result))
    return " ".join(result)
    
def translateRules(regles1,MGL=False):
    results=[]
    for motif,context in regles1.items():
        results.append(translateRule(motif,context))

    pattern = r"(\\phonfeat(?:\[[^\]]*\])?\{[^}]*\}(?:(?!\\phonfeat).)*)$"
    match = re.search(pattern, " ".join(results))

    if MGL and match:
        # print(match.group(0))
        return match.group(0)
    else:
        return " ".join(results)

In [159]:
print('r"'+translateRule("E-â","[aE][mnl]E")+'"')
# feat2phonrule("[lk]")

r"\phonfeat[l]{+son\\+syl\\-cons\\-ant\\-cor\\-back\\-high\\-round\\-ATR\\-nas\\-lat\\+cont\\+voice\\-strid} \phonfeat[l]{+son\\-syl\\+cons\\+ant\\-back\\-high\\-low\\-round\\-ATR\\-cont\\+voice} E"


In [190]:
case1,case2="inf","pP"


tPaires=[
    ("kalE","kalâ"),
    ("kalmE","kalmâ"),
    ("pasE","pasâ"),
    ("parlE","parlâ"),
    ("pâsE","pâsâ"),
    ("truvE","truvâ"),
    ("lEsE","lEsâ"),
    ("arivE","arivâ"),
    ("dOnE","dOnâ"),
    ("r6gardE","r6gardâ"),
    ("rEstE","rEstâ"),
    ("arEtE","arEtâ"),
    ("tyHE","tyHâ"),
    ("d6mâdE","d6mâdâ"),
    ("SErSE","SErSâ"),
    ("sEdE","sEdâ"),
       ]

paires=tPaires[:]

lCombinaisons=[]
for r in range(len(paires)+1):
    lCombinaisons.extend(combinations(paires,r))

sRegles=set()
for c in lCombinaisons:
    # print(c)
    patrons=pairePatrons(case1,case2)
    classes=paireClasses(case1,case2)
    for (f1,f2) in c:
        # print(f1,f2)
        patrons.ajouterFormes(f1,f2,diff(f1,f2))
    (regles1,regles2)=patrons.calculerGM()
    # print(regles1)
    sRegles.add(translateRules(regles1,MGL=False))
len(paires),len(lCombinaisons),len(sRegles)

(16, 65536, 4063)

In [194]:
patrons=pairePatrons(case1,case2)
classes=paireClasses(case1,case2)
for (f1,f2) in tPaires:
    patrons.ajouterFormes(f1,f2,diff(f1,f2))
(regles1,regles2)=patrons.calculerGM()
print(regles1)
print(translateRules(regles1,MGL=False))

{'E-â': '^(.*[ptkbdgfsSvzZmnJNjlrE6aêâ][ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwH])E$'}
^(.* \phonfeat[l]{-round\\-ATR} \phonfeat[l]{} \phonfeat[l]{-syl\\-low\\-ATR} )E$


In [191]:
sRegles

{'',
 '^(.* \\phonfeat[l]{+son\\\\-syl\\\\+cons\\\\-high\\\\-low\\\\-round\\\\-ATR\\\\-nas\\\\+voice\\\\+strid} \\phonfeat[l]{+son\\\\+syl\\\\-cons\\\\-ant\\\\-cor\\\\-back\\\\-low\\\\-round\\\\-nas\\\\-lat\\\\+cont\\\\+voice\\\\-strid} \\phonfeat[l]{-son\\\\-syl\\\\+cons\\\\+ant\\\\-back\\\\-high\\\\-low\\\\-round\\\\-ATR\\\\-nas\\\\-lat\\\\+cont\\\\+strid} )E$',
 '^( \\phonfeat[l]{-son\\\\-syl\\\\+cons\\\\-low\\\\-round\\\\-ATR\\\\-nas\\\\-lat\\\\-cont\\\\-voice\\\\-strid} \\phonfeat[l]{+son\\\\+syl\\\\-cons\\\\-ant\\\\-cor\\\\-lat\\\\+cont\\\\+voice\\\\-strid} \\phonfeat[l]{-syl\\\\-back\\\\-low\\\\-ATR\\\\-nas} )E$',
 '^(.* \\phonfeat[l]{-high\\\\-round\\\\-ATR\\\\-nas\\\\-lat} \\phonfeat[l]{+son\\\\-ant\\\\-cor\\\\-low\\\\-nas\\\\-lat\\\\+cont\\\\+voice} \\phonfeat[l]{-syl\\\\+cons\\\\+ant\\\\-back\\\\-high\\\\-low\\\\-round\\\\-ATR} )E$',
 '^(.* \\phonfeat[l]{-high\\\\-round\\\\-ATR\\\\-nas\\\\-lat} \\phonfeat[l]{+son\\\\-ant\\\\-cor\\\\-nas\\\\-lat\\\\+cont\\\\+voice} \\phonfeat

In [52]:
for i, (f1,f2) in enumerate(paires):
    print(", ".join([c1+"-"+c2 for (c1,c2) in paires[:i+1]]))
    patrons.ajouterFormes(f1,f2,diff(f1,f2))
    (regles1,regles2)=patrons.calculerGM()
    translateRule(regles1)
    print()
    print()

alE-alâ
E-â ^alE$
['^alE$']
E-â : ^alE$

alE-alâ, parlE-parlâ
E-â ^(.*[rE6a])lE$
['^(.*', '[rE6a]', ')lE$']
E-â : ^(.*('+son', '-ant', '-cor', '-high', '-round', '-ATR', '-nas', '-lat', '+cont', '+voice'))lE$

alE-alâ, parlE-parlâ, EmE-Emâ
E-â ^(.*[rE6a][mnl])E$
['^(.*', '[rE6a]', '[mnl]', ')E$']
E-â : ^(.*('+son', '-ant', '-cor', '-high', '-round', '-ATR', '-nas', '-lat', '+cont', '+voice')('+son', '-syl', '+cons', '+ant', '-back', '-high', '-low', '-round', '-ATR', '-cont', '+voice'))E$

alE-alâ, parlE-parlâ, EmE-Emâ, pasE-pasâ
E-â ^(.*[rE6a][ptbdfsvzmnl])E$
['^(.*', '[rE6a]', '[ptbdfsvzmnl]', ')E$']
E-â : ^(.*('+son', '-ant', '-cor', '-high', '-round', '-ATR', '-nas', '-lat', '+cont', '+voice')('-syl', '+cons', '+ant', '-back', '-high', '-low', '-round', '-ATR'))E$

alE-alâ, parlE-parlâ, EmE-Emâ, pasE-pasâ, pâsE-pâsâ
E-â ^(.*[rE6aêâ][ptbdfsvzmnl])E$
['^(.*', '[rE6aêâ]', '[ptbdfsvzmnl]', ')E$']
E-â : ^(.*('+son', '-ant', '-cor', '-high', '-round', '-ATR', '-lat', '+cont', '+voice')('

In [ ]:
p=("ii3S","ppMS")
k="wajE-y"
v=regles[p].patrons[k]

In [ ]:
print(k,v)
m=re.split("(\[[^\]]*\])",v)
print(k,end=" : ")
for e in m:
    if e.startswith("[") and e.endswith("]"):
        print(fs.lattice[e[1:-1]].intent,end="")
    elif e!="":
        print(e,end="")

In [ ]:
regles[p].sortirForme("brwajE")

In [261]:
from scipy.stats import entropy

entropy([286, 43, 2, 32, 11, 2])

0.8248065676401228